In [9]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegressionCV, LogisticRegression
from sklearn.decomposition import PCA
from sklearn.model_selection import cross_val_score

# === Load datasets ===
folder = r"C:\Users\exm499\Desktop\EK\PFAS_classification\Important feature"
label_df = pd.read_excel(f"{folder}\\label.xlsx", index_col=0)
deg_df = pd.read_excel(f"{folder}\\DEG_fold.xlsx", index_col=0)
toxcast_df = pd.read_excel(f"{folder}\\PFAS_HitCall_Matrix.xlsx", index_col=0)
chemotype_df = pd.read_excel(f"{folder}\\Chemotype_15PFAS_1.xlsx", index_col=0)
y = label_df["Group"]

# === LASSO Logistic Regression with CV Accuracy ===
def run_lasso(X, y):
    X = X.loc[:, y.index].T
    feature_names = X.columns

    model = LogisticRegressionCV(cv=5, penalty='l1', solver='liblinear', max_iter=2000)
    model.fit(X, y)
    train_accuracy = model.score(X, y)

    # Cross-validation accuracy
    cv_model = LogisticRegression(penalty='l1', solver='liblinear', max_iter=2000)
    scores = cross_val_score(cv_model, X, y, cv=5)
    cv_mean = np.mean(scores)
    cv_std = np.std(scores)

    # Feature selection
    coef = np.abs(model.coef_).sum(axis=0)
    mask = coef != 0
    selected_features = feature_names[mask]

    return {
        "train_accuracy": train_accuracy,
        "cv_mean": cv_mean,
        "cv_std": cv_std,
        "selected_features": list(selected_features)
    }

# === Logistic Regression on PCA ===
def run_lr_pca(X, y, n_components=5):
    X = X.loc[:, y.index].T
    pca = PCA(n_components=n_components)
    X_pca = pca.fit_transform(X)
    model = LogisticRegression(max_iter=2000)
    scores = cross_val_score(model, X_pca, y, cv=5)
    return {"mean_accuracy": np.mean(scores), "std_accuracy": np.std(scores), "n_components": n_components}

# === Run evaluations ===
for name, df in [("DEG", deg_df), ("ToxCast", toxcast_df), ("Chemotype", chemotype_df)]:
    print(f"\n--- {name} ---")
    res_lasso = run_lasso(df, y)
    res_pca = run_lr_pca(df, y, n_components=5)

    print(f"LASSO Train Accuracy:   {res_lasso['train_accuracy']:.3f}")
    print(f"LASSO CV Accuracy:      {res_lasso['cv_mean']:.3f} ± {res_lasso['cv_std']:.3f} (Selected: {len(res_lasso['selected_features'])})")
    print(f"LR-PCA CV Accuracy:     {res_pca['mean_accuracy']:.3f} ± {res_pca['std_accuracy']:.3f}")


--- DEG ---


ValueError: Input X contains NaN.
LogisticRegressionCV does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

In [10]:
import pandas as pd

# Load the data
deg_df = pd.read_excel(r"DEG_fold.xlsx", index_col=0)

# Check for any NaNs in the entire dataframe
print("Any NaNs?", deg_df.isna().any().any())

# How many total NaNs?
total_nans = deg_df.isna().sum().sum()
print(f"🔍 Total missing values (NaNs): {total_nans}")

# Show genes with any missing value
missing_genes = deg_df[deg_df.isna().any(axis=1)]
print(f"🧬 Genes with NaNs: {missing_genes.shape[0]}")
print(missing_genes.head())

FileNotFoundError: [Errno 2] No such file or directory: 'DEG_fold.xlsx'

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

# === Evaluation Function (Accuracy Only) ===
def evaluate_model_accuracy(X, y, model, name="Model"):
    X = X.loc[:, y.index].T  # Align and transpose: samples × features

    scores = cross_val_score(model, X, y, cv=5)
    print(f"{name:30} Accuracy: {np.mean(scores):.3f} ± {np.std(scores):.3f}")

# === Models ===
models = {
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Neural Net": MLPClassifier(hidden_layer_sizes=(100,), max_iter=1000, random_state=42),
    "SVM": SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=42),
    "Logistic Regression": LogisticRegression(max_iter=2000, solver='lbfgs', multi_class='auto')
}

# === Run Evaluation ===
datasets = {
    "DEG": deg_df,
    "ToxCast": toxcast_df,
    "Chemotype": chemotype_df
}

for dataset_name, dataset in datasets.items():
    for model_name, model in models.items():
        evaluate_model_accuracy(dataset, y, model, name=f"{model_name} on {dataset_name}")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score

def compute_feature_r2(X, y, top_n=10, dataset_name="Dataset", verbose=True):
    X = X.loc[:, y.index].T  # align samples
    y_encoded = LabelEncoder().fit_transform(y)

    r2_values = []
    for feature in X.columns:
        x_col = X[[feature]].values  # single feature as column
        model = LinearRegression().fit(x_col, y_encoded)
        y_pred = model.predict(x_col)
        r2 = r2_score(y_encoded, y_pred)
        r2_values.append((feature, r2))

    r2_df = pd.DataFrame(r2_values, columns=['Feature', 'R2'])
    r2_df = r2_df.sort_values(by="R2", ascending=False).reset_index(drop=True)

    if verbose:
        print(f"\n📈 Top {top_n} Features by R² for {dataset_name}")
        print(r2_df.head(top_n).to_string(index=False, float_format="%.3f"))

    return r2_df

# Run for each dataset
deg_r2 = compute_feature_r2(deg_df, y, top_n=50, dataset_name="DEG")
tox_r2 = compute_feature_r2(toxcast_df, y, top_n=10, dataset_name="ToxCast")
chem_r2 = compute_feature_r2(chemotype_df, y, top_n=10, dataset_name="Chemotype")

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.spatial.distance import pdist
import matplotlib.pyplot as plt

res_lasso_deg = run_lasso(deg_df, y)
selected_lasso_genes = res_lasso_deg["selected_features"]

def plot_dendrogram_cosine(selected_genes, deg_df, y, title):
    X = deg_df.loc[selected_genes, y.index].T  # samples × genes
    distance_matrix = pdist(X, metric='cosine')
    linked = linkage(distance_matrix, method='ward')

    plt.figure(figsize=(10, 4))
    dendrogram(linked, labels=y.index.tolist(), leaf_rotation=90)
    plt.title(f"Dendrogram - {title} (Cosine Distance)")
    plt.xlabel("Sample")
    plt.ylabel("Linkage Distance")
    plt.tight_layout()
    plt.show()

# If you've already run compute_high_r2_features():
selected_rf_genes = deg_high_r2["Feature"].tolist()

# Plot both
plot_dendrogram_cosine(selected_lasso_genes, deg_df, y, "LASSO-Selected Genes (130)")
plot_dendrogram_cosine(selected_rf_genes, deg_df, y, "Random Forest / R²-Selected Genes (151)")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score

# Function to compute R² per feature (no scaling)
def compute_high_r2_features(X, y, threshold=0.6):
    X = X.loc[:, y.index].T  # transpose: samples × features
    y_encoded = LabelEncoder().fit_transform(y)

    r2_values = []
    for feature in X.columns:
        x_col = X[[feature]].values
        model = LinearRegression().fit(x_col, y_encoded)
        y_pred = model.predict(x_col)
        r2 = r2_score(y_encoded, y_pred)
        r2_values.append((feature, r2))

    # Create DataFrame and filter
    r2_df = pd.DataFrame(r2_values, columns=['Feature', 'R2'])
    filtered_df = r2_df[r2_df["R2"] > threshold].sort_values(by="R2", ascending=False).reset_index(drop=True)

    print(f"\n✅ Number of features with R² > {threshold}: {len(filtered_df)}")
    return filtered_df

# Run for DEG dataset
deg_high_r2 = compute_high_r2_features(deg_df, y, threshold=0.6)

In [ ]:
from sklearn.model_selection import cross_val_score
# Function to compute R² per feature (no scaling)
def compute_high_r2_features(X, y, threshold=0.6):
    X = X.loc[:, y.index].T  # transpose: samples × features
    y_encoded = LabelEncoder().fit_transform(y)

    r2_values = []
    for feature in X.columns:
        x_col = X[[feature]].values
        model = LinearRegression().fit(x_col, y_encoded)
        y_pred = model.predict(x_col)
        r2 = r2_score(y_encoded, y_pred)
        r2_values.append((feature, r2))

    # Create DataFrame and filter
    r2_df = pd.DataFrame(r2_values, columns=['Feature', 'R2'])
    filtered_df = r2_df[r2_df["R2"] > threshold].sort_values(by="R2", ascending=False).reset_index(drop=True)

    cv_scores = cross_val_score(model, X, y_encoded, cv=5)
    
    print(f"\n✅ Number of features with R² > {threshold}: {len(filtered_df)}")
    return filtered_d

print(f"Cross-validated accuracy: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score
from tqdm import tqdm
from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.spatial.distance import pdist
import matplotlib.pyplot as plt

# === Step 1: Feature selection using Random Forest R² ===
def compute_high_r2_features_rf(X, y, threshold=0.6, verbose=True):
    # Check orientation
    if X.shape[0] > X.shape[1]:
        print(f"✅ Interpreting X as [genes × samples]: shape = {X.shape}")
        X = X.loc[:, y.index].T  # now: samples × genes
    else:
        print(f"✅ Assuming X is already [samples × genes]: shape = {X.shape}")

    y_encoded = LabelEncoder().fit_transform(y)

    r2_values = []
    print("🔄 Computing R² values per feature using Random Forest...")

    for feature in tqdm(X.columns, desc="Evaluating features"):
        x_col = X[[feature]].values
        model = RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=42)
        model.fit(x_col, y_encoded)
        y_pred = model.predict(x_col)
        r2 = r2_score(y_encoded, y_pred)
        r2_values.append((feature, r2))

    r2_df = pd.DataFrame(r2_values, columns=["Feature", "R2"])
    filtered_df = r2_df[r2_df["R2"] > threshold].sort_values(by="R2", ascending=False).reset_index(drop=True)

    # Cross-validation on selected features
    X_selected = X[filtered_df["Feature"]]
    model_cv = RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=42)
    cv_scores = cross_val_score(model_cv, X_selected, y_encoded, cv=5, scoring='r2')

    print(f"\n✅ Number of features with R² > {threshold}: {len(filtered_df)}")
    print(f"📈 Cross-validated R²: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

    return filtered_df

# Select genes with high RF R²
deg_high_r2_rf = compute_high_r2_features_rf(deg_df, y, threshold=0.6)
selected_genes = deg_high_r2_rf["Feature"].tolist()

# === Step 2: Cosine-based Dendrogram ===
def plot_dendrogram_cosine(selected_genes, df, y, title):
    X = df.loc[selected_genes, y.index].T
    dist = pdist(X, metric='cosine')
    linked = linkage(dist, method='ward')

    plt.figure(figsize=(10, 4))
    dendrogram(linked, labels=y.index.tolist(), leaf_rotation=90)
    plt.title(f"Dendrogram - {title} (Cosine Distance)")
    plt.xlabel("Sample")
    plt.ylabel("Linkage Distance")
    plt.tight_layout()
    plt.show()

# Plot dendrogram
plot_dendrogram_cosine(selected_genes, deg_df, y, "Random Forest R² > 0.6")